# Pre-entrega 3: demo del sistema RAG local

Este notebook recorre el flujo End-to-End importando los módulos de `src/` (no duplica lógica):

1. **Ingesta**: chunking con `RecursiveCharacterTextSplitter` y persistencia en ChromaDB (se omite si la base ya existe).
2. **Recuperación**: la pregunta se convierte en embedding y se traen los `k=4` fragmentos más cercanos.
3. **Generación grounded**: cadena LCEL asíncrona que termina en `PydanticOutputParser`.
4. **Fuera de contexto**: el sistema responde "No lo sé".
5. **Asincronía**: varias preguntas en paralelo con `asyncio.gather`.

El dataset (`data/`) es documentación interna **inventada** de una empresa ficticia (Transportes Altiplano y su plataforma Kairo), así el modelo no puede responder con conocimiento propio.

In [1]:
import logging
import time
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm sin ipywidgets
load_dotenv()
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s | %(message)s")
logging.getLogger("rag").setLevel(logging.INFO)

from src import config
from src.chain import ask, ask_many, get_chain
from src.ingest import build_vectorstore
from src.retriever import get_vectorstore

print(f"Modelo de embeddings: {config.EMBEDDING_MODEL}")
print(f"chunk_size={config.CHUNK_SIZE}, chunk_overlap={config.CHUNK_OVERLAP}, top_k={config.TOP_K}")

Modelo de embeddings: intfloat/multilingual-e5-small
chunk_size=800, chunk_overlap=100, top_k=4


## 1. Ingesta

`build_vectorstore()` primero verifica si `./vectorstore` existe y tiene la colección poblada. Si es así, no re-indexa (recalcular los embeddings es lo más caro de todo el proceso). Para reconstruirla se usa `force=True` o `python main.py ingest --force`.

In [2]:
resultado = build_vectorstore()
resultado

INFO rag | Base ya indexada en ./vectorstore (34 fragmentos): se omite la ingesta


ResultadoIngesta(indexada=False, documentos=0, fragmentos=34)

Si se vuelve a llamar, la ingesta se omite:

In [3]:
build_vectorstore()

INFO rag | Base ya indexada en ./vectorstore (34 fragmentos): se omite la ingesta


ResultadoIngesta(indexada=False, documentos=0, fragmentos=34)

## 2. Recuperación sola (sin LLM)

Antes de sumar el LLM conviene validar que los fragmentos recuperados tengan sentido. La distancia es coseno: más baja significa más parecido.

In [4]:
def buscar(consulta: str) -> None:
    print(f"Consulta: {consulta}")
    for doc, distancia in get_vectorstore().similarity_search_with_score(consulta, k=config.TOP_K):
        print(f"  {distancia:.3f}  {doc.metadata['source']}#{doc.metadata['chunk_index']}  ({doc.metadata['seccion']})")


buscar("¿Cómo se calcula el peso facturable de un envío?")
print()
buscar("¿Cuál es la capital de Australia?")

INFO rag | Cargando el modelo de embeddings intfloat/multilingual-e5-small


Consulta: ¿Cómo se calcula el peso facturable de un envío?


  0.092  04_api_cotizaciones.md#4  (Cómo se calcula el precio)
  0.151  04_api_cotizaciones.md#2  (Endpoints principales)
  0.165  01_arquitectura_kairo.md#3  (Servicios principales)
  0.181  04_api_cotizaciones.md#3  (Reservar un envío)

Consulta: ¿Cuál es la capital de Australia?
  0.232  01_arquitectura_kairo.md#9  (Decisiones de arquitectura)
  0.234  04_api_cotizaciones.md#2  (Endpoints principales)
  0.244  01_arquitectura_kairo.md#1  (Qué es Kairo)
  0.244  01_arquitectura_kairo.md#8  (Observabilidad)


En la pregunta del dataset, el fragmento correcto ("Cómo se calcula el precio") queda primero y con una distancia clara. En la pregunta de cultura general, el retriever igual devuelve 4 fragmentos (siempre devuelve `k`), pero todos son irrelevantes y con distancias más altas. Por eso la decisión final de decir "No lo sé" la toma el prompt de veracidad, no el retriever.

## 3. Pregunta respondida con el contexto

`ask()` hace `await rag_chain.ainvoke(pregunta)` y devuelve un objeto `RAGAnswer` ya validado por el `PydanticOutputParser`.

In [5]:
respuesta = await ask("¿Qué estrategia de despliegue se usa en producción y qué porcentaje del tráfico recibe primero la versión nueva?")
print(respuesta.model_dump_json(indent=2))

INFO rag | Pregunta: ¿Qué estrategia de despliegue se usa en producción y qué porcentaje del tráfico recibe primero la versión nueva?


INFO rag | Recuperados 4 fragmentos: 02_guia_despliegue.md#2, 02_guia_despliegue.md#0, 01_arquitectura_kairo.md#4, 02_guia_despliegue.md#1


INFO rag | Llamada 1 al modelo claude-opus-5-5


INFO rag | Respuesta del modelo: fin=end_turn | tokens entrada=2255 salida=282


INFO rag | Respuesta validada: found_in_context=True, fuentes=['02_guia_despliegue.md']


{
  "answer": "En producción los servicios se despliegan con la estrategia canary, usando Argo Rollouts. Primero, la versión nueva recibe el 10 % del tráfico durante 15 minutos. Si la tasa de errores no supera el 1 % y la latencia p99 no sube más de un 20 %, se pasa al 50 % durante otros 15 minutos, y si las métricas siguen sanas se completa al 100 %. Si alguna métrica supera los umbrales, Argo Rollouts revierte automáticamente a la versión anterior y deja un aviso en #kairo-deploys. El servicio despacho usa un umbral más estricto: la tasa de errores no puede superar el 0,5 %.",
  "found_in_context": true,
  "sources": [
    "02_guia_despliegue.md"
  ],
  "confidence": "alta"
}


## 4. Preguntas fuera del contexto

Una de cultura general y una sobre la empresa que la documentación no cubre. En las dos el modelo podría inventar algo plausible; el filtro de veracidad lo obliga a responder "No lo sé", con `found_in_context=false` y sin fuentes.

In [6]:
for pregunta in ["¿Cuál es la capital de Australia?", "¿Qué CRM usa el equipo comercial de Transportes Altiplano?"]:
    respuesta = await ask(pregunta)
    print(pregunta)
    print(respuesta.model_dump_json(indent=2), end="\n\n")

INFO rag | Pregunta: ¿Cuál es la capital de Australia?


INFO rag | Recuperados 4 fragmentos: 01_arquitectura_kairo.md#9, 04_api_cotizaciones.md#2, 01_arquitectura_kairo.md#1, 01_arquitectura_kairo.md#8


INFO rag | Llamada 1 al modelo claude-opus-5-5


INFO rag | Respuesta del modelo: fin=end_turn | tokens entrada=2333 salida=60


INFO rag | Respuesta validada: found_in_context=False, fuentes=[]


INFO rag | Pregunta: ¿Qué CRM usa el equipo comercial de Transportes Altiplano?


... (2 outputs left out)

¿Cuál es la capital de Australia?
{
  "answer": "No lo sé. No tengo acceso a esa información en la documentación disponible.",
  "found_in_context": false,
  "sources": [],
  "confidence": "baja"
}



INFO rag | Respuesta del modelo: fin=end_turn | tokens entrada=2207 salida=60


INFO rag | Respuesta validada: found_in_context=False, fuentes=[]


¿Qué CRM usa el equipo comercial de Transportes Altiplano?
{
  "answer": "No lo sé. No tengo acceso a esa información en la documentación disponible.",
  "found_in_context": false,
  "sources": [],
  "confidence": "baja"
}



## 5. Asincronía: secuencial vs. paralelo

Las mismas 4 preguntas, primero una por una y después todas juntas con `asyncio.gather` (`ask_many`). Como la mayor parte del tiempo es espera de red al LLM, en paralelo el total se acerca al de la pregunta más lenta.

In [7]:
logging.getLogger("rag").setLevel(logging.WARNING)  # menos logs para esta comparación
get_chain()  # la cadena (y el modelo de embeddings) ya están cargados: se mide solo la consulta

preguntas = [
    "¿En cuánto tiempo hay que reconocer una alerta SEV1?",
    "¿Cuánto dura el token de acceso de la API?",
    "¿Qué días no se puede desplegar a producción?",
    "¿Quién inventó la bombilla eléctrica?",
]

inicio = time.perf_counter()
secuenciales = [await ask(p) for p in preguntas]
t_secuencial = time.perf_counter() - inicio

inicio = time.perf_counter()
paralelas = await ask_many(preguntas)
t_paralelo = time.perf_counter() - inicio

for pregunta, r in zip(preguntas, paralelas):
    estado = "EN CONTEXTO" if r.found_in_context else "FUERA"
    print(f"[{estado:11}] {pregunta}\n              -> {r.answer}  {r.sources}\n")
print(f"Secuencial: {t_secuencial:.1f} s | Paralelo (asyncio.gather): {t_paralelo:.1f} s")

[EN CONTEXTO] ¿En cuánto tiempo hay que reconocer una alerta SEV1?
              -> Una alerta SEV1 tiene un tiempo de respuesta de 10 minutos. Ese tiempo se mide desde que suena la alerta hasta que alguien la reconoce en Sereno. Si la guardia principal no la reconoce en ese plazo, Sereno escala automáticamente a la guardia secundaria y, cinco minutos después, al responsable del equipo.  ['03_politica_guardias.md']

[EN CONTEXTO] ¿Cuánto dura el token de acceso de la API?
              -> El token de acceso dura 60 minutos. Se obtiene en el endpoint `POST /oauth/token` con el flujo client credentials de OAuth 2.0, usando el identificador de cliente y la clave secreta. Después se envía en el encabezado `Authorization: Bearer <token>`.  ['04_api_cotizaciones.md']

[EN CONTEXTO] ¿Qué días no se puede desplegar a producción?
              -> No se despliega a producción en estas ventanas de congelamiento:
- Desde el viernes a las 15:00 hasta el lunes a las 09:00 (hora de Argentina).
- Del 

## Conclusión

- La base se indexa una sola vez y se reutiliza; la misma instancia de embeddings (`get_embeddings()`) sirve para indexar y para consultar, y la colección guarda el nombre del modelo para detectar si alguien intenta consultar con otro.
- Con `k=4` el modelo recibe poco contexto, todo relevante (Lost in the Middle).
- Toda respuesta pasa por el `PydanticOutputParser`: o viene del contexto con sus fuentes, o es un "No lo sé" sin fuentes. El esquema rechaza cualquier combinación intermedia y la cadena reintenta.